# 00 · Quickstart: from gas fields to transmitted flux and P1D

SkewerJAX computes the optical depth $\tau(v)$ and the transmitted flux $F = e^{-\tau}$ of many transitions along
skewers, and their summary statistics. This notebook runs the whole chain on a toy box in five steps:

1. **lines** (`LineBundle`) and **cosmology** (`Cosmology`);
2. **skewers** (`SkewerBundle`);
3. **gas cells** (`Cells` of a Gaussian random field);
4. **$\tau$ and $F$** (`compute_optical_depth`);
5. **statistics** (`OverfluxStatistics1D`).

The graph below shows how these ingredients depend on each other. Every later notebook opens one of them; its number
is under each box. `00_quickstart_step_by_step` is the second entry point: the same chain on a simulation snapshot,
one figure per step, with the equations. Runtime: ~30 s on a CPU.

**Requires:** nothing: it builds every ingredient once, on a toy box.  
**Independent of:** —

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import use_style, use_backend
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()          # paper style (white); use_style("dark") for the dark one: same colour-blind-safe palette in both
from skewerjax.utils.plotting import (ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, BG, DIM, CMAPS, LABELS,
                                      FIGSIZES, add_panel_letters, plot_dag, plot_spectra, zoom_inset)

In [ ]:
plot_dag(('lines', 'cosmology', 'skewers', 'samplers', 'fields', 'tau', 'overflux', 'stats',)); plt.show()

**1–2. Lines, cosmology and skewers.** $8 \times 8$ parallel skewers run through a 50 cMpc box whose far end sits at
$z = 2.2$. `SkewerBundle` solves $a(x)$ along every skewer (review notes, Eq. `dadx_general`).

In [ ]:
from skewerjax.core import (LineBundle, Cosmology, SkewerBundle, grid_skewer_points, phys, PowerLawGRF, grf_cells,
                            compute_optical_depth, wrap_to_ref_line, DeltaFieldBuilder)
from skewerjax.core.stats import OverfluxStatistics1D

lines = LineBundle(["HI-LyA", "SiIII-1206"], ref_line_id="HI-LyA")
cosmo = Cosmology(H0=phys.H0, Omega_r_0=0.0, Omega_m_0=1.0, Omega_k_0=0.0, Omega_lambda_0=0.0)
L, z = 50.0, 2.2                                                      # box [cMpc], redshift of the far end
points = grid_skewer_points(L, 8, 2048, direction="y").reshape(64, 2048, 3)
skewers = SkewerBundle(spatial_points=points, cosmologies=cosmo, a0=1 / (1 + z), x0=L)
print(f"{skewers.N_sk} skewers x {skewers.N_ppsk} points; a from {float(skewers.scale_factors.min()):.4f} to {float(skewers.scale_factors.max()):.4f}")

**3. Gas cells.** One Gaussian random field $\delta$ (`PowerLawGRF`, $\sigma_\delta = 0.2$) sets the density of both
species on the cells of its grid (`grf_cells`, notebook 04), $n = \bar n\,(1 + \delta)$, with Si III 50 times less
abundant. The temperature is $T = 1.5\times10^4$ K and there are no peculiar velocities.

In [ ]:
grf = PowerLawGRF(A_s=0.04, k_0=2.0, n_s=-1.25, L=(L,) * 3, N=(64,) * 3)          # sigma_delta = 0.2
key = jax.random.PRNGKey(0)
delta = np.asarray(grf.sample(key))
cells = grf_cells(delta, grf, {"HI": 1e-12, "SiIII": 1e-12 / 5}, temperature=1.5e4)     # n = n_bar (1 + delta) [comoving cm^-3]
print(f"sigma_delta = {float(jnp.sqrt(grf.variance())):.2f}, min(1 + delta) = {1 + delta.min():.3f}; {cells}")

**4. $\tau$ and $F$.** Both lines are evaluated on the $\mathrm{Ly}\alpha$ axis (`line_frame="reference"`), where Si III appears
2270 km/s bluer. The spectra are periodic over the box (`velocity_period=True`), on a grid of one period, because the
P1D estimator below is a discrete Fourier transform. The warning is the sampling check:
the narrow Si III lines ($b \approx 3$ km/s) are under-sampled at 1.7 km/s per point (notebook 06).

In [ ]:
a_dagger = 1 / (1 + z)
v_max = float(phys.c_kms * jnp.log(a_dagger / skewers.scale_factors[0, 0, 0]))
v = (jnp.arange(512) + 0.5) * v_max / 512                                    # one period of the box
tau = compute_optical_depth(skewers, v, lines, cells, a_dagger=a_dagger, velocity_period=True, line_frame="reference")
F = np.exp(-np.asarray(tau))
print("tau", tau.shape, "| <F>:", {l.id: round(float(F[i].mean()), 3) for i, l in enumerate(lines.lines)})

**5. Statistics.** `DeltaFieldBuilder` forms $\delta_F$ of each line and of their product, and `OverfluxStatistics1D`
gives P1D and $\xi_{\rm 1D}$.

In [ ]:
stats = OverfluxStatistics1D(DeltaFieldBuilder(tau, labels=["Lya", "SiIII"], key_joiner="+").build(), float(v[1] - v[0])).compute_p1d()
k = np.asarray(stats.k)[1:]
fig, axs = plt.subplots(1, 3, figsize=(22, 6), layout="constrained")
from skewerjax.core import deposit, segment_fields, line_elements
pts0 = np.asarray(skewers.spatial_points[:1], dtype=float)                   # skewer 0: the density of the cells it crosses
used, seg = deposit(cells, pts0)
w0 = np.asarray(line_elements(jnp.asarray(pts0)))
for j, sp in enumerate(lines.species_keys):
    n = segment_fields(seg, used.species[sp]["number_density"], used.species[sp]["temperature"], 0.0)["column_length"][0] / w0[0]
    axs[0].semilogy(pts0[0, :, 1], n, color=EC[sp], **(MS["default"] if j == 0 else MS["matched"]), label=sp)
axs[0].set(xlabel="y [cMpc]", ylabel=r"$n\ [\mathrm{cm^{-3}}]$", title="gas along skewer 0")
for j, l in enumerate(lines.lines):
    axs[1].plot(v, F[j, 0], color=EC[l.id], label=l.id)
axs[1].plot(v, F[:, 0].prod(0), color=FG, lw=1.2, label="total")
axs[1].set(xlabel=r"$v$ [km/s] (Ly$\alpha$ frame)", ylabel="F", title="transmitted flux")
axs[2].loglog(k, k * np.asarray(stats.Pk_total)[1:] / np.pi, color=FG, label="total")
axs[2].loglog(k, k * np.asarray(stats.Pk_pair[("Lya", "Lya")])[1:] / np.pi, color=EC["HI"], **MS["matched"], label=r"Ly$\alpha$ alone")
axs[2].set(xlabel=r"$k$ [s/km]", ylabel=r"$k P_{\rm 1D}/\pi$", title="P1D over 64 skewers")
for ax in axs: ax.legend()
add_panel_letters(axs); fig.suptitle("Toy box: one density field, two lines, their flux and its 1D power spectrum", fontsize=22)
plt.show()

- (b) Si III is a weak copy of the same forest, shifted by $\Delta v = 2270$ km/s on the $\mathrm{Ly}\alpha$ axis, and wrapped
  around the box because the spectrum was asked to be periodic. The default integral is not periodic: notebook 06,
  §6, explains the option.
- (c) At this abundance the total P1D is barely above $\mathrm{Ly}\alpha$ alone. The $\mathrm{Ly}\alpha \times$ Si III cross term
  oscillates with period $2\pi/\Delta v$ in $k$, and notebook 07 measures it term by term.

The next notebooks open each ingredient in the order of the graph:

- lines (01), cosmology (02) and skewers (03);
- fields and samplers (04), and simulation outputs (05);
- line profiles and $\tau$ (06), and summary statistics (07);
- the comparison with an external reference code (08).

`00_quickstart_step_by_step` follows one simulated skewer from the gas to the statistics.